# Capítulo 6 — Otimização de Inferência: LMCache e Speculative Decoding

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aejepsen/ebook-llm-on-premise/blob/main/notebooks/cap06_otimizacao.ipynb)

**Fontes originais:**
- [orca3/llm-model-serving — ch07/LMCache.ipynb](https://github.com/orca3/llm-model-serving/tree/main/ch07)
- [orca3/llm-model-serving — ch07/SpecDecode.ipynb](https://github.com/orca3/llm-model-serving/tree/main/ch07)

Adaptado e comentado por **Anderson Ejepsen**.

Créditos aos autores originais do repositório [orca3/llm-model-serving](https://github.com/orca3/llm-model-serving).

> **Requisitos:** GPU A100 80GB recomendada. No Colab, selecione A100.

## Configuração do Ambiente

In [ ]:
!uv pip install lmcache==0.3.5 vllm==0.10.1 torch

In [ ]:
import torch
import lmcache
import vllm
from importlib.metadata import version
print(f"lmcache: {version('lmcache')}")
print(f"vllm: {version('vllm')}")
print(f"torch: {version('torch')}")
!nvidia-smi

### Funções utilitárias

Funções para verificar a saúde do servidor vLLM e monitorar logs.

In [ ]:
import time
import requests
from collections import deque

def tail_log(filename, n=2):
    """Lê e imprime as últimas n linhas de um arquivo de log."""
    try:
        with open(filename, 'r') as f:
            last_lines = deque(f, maxlen=n)
        print(f"\nÚltimas {n} linhas de {filename}:")
        for line in last_lines:
            print(line.rstrip())
    except FileNotFoundError:
        print(f"Arquivo de log {filename} não encontrado.")
    except Exception as e:
        print(f"Erro ao ler arquivo de log: {e}")

def check_vllm_status():
    """Aguarda o servidor vLLM ficar saudável."""
    max_retries = 100
    wait_time = 10

    print("Aguardando servidor vLLM...")

    for i in range(max_retries):
        try:
            r = requests.get("http://127.0.0.1:8000/health")
            if r.ok:
                print("Servidor vLLM pronto!")
                tail_log("vllm.log", n=2)
                break
        except Exception:
            print(f"Tentativa {i+1}: Ainda não está pronto...")
        tail_log("vllm.log", n=2)
        time.sleep(wait_time)
    else:
        print("Timeout ao aguardar servidor vLLM.")
        tail_log("vllm.log", n=2)

---
## 6.1 — LMCache: Cache de KV para Prompts Repetidos

O **LMCache** é uma extensão do vLLM que armazena os vetores Key/Value (KV) computados durante o prefill na memória CPU. Quando o mesmo prefixo de prompt é reutilizado, o KV cache é restaurado em vez de recomputado, **eliminando a fase de prefill**.

Isso é particularmente útil para:
- Prompts de sistema longos que são reutilizados entre requisições
- RAG com contexto compartilhado
- Aplicações multi-turno com histórico

### Iniciando o vLLM com LMCache

Configuramos o LMCache com os seguintes parâmetros:
- `LMCACHE_CHUNK_SIZE=256`: tamanho dos chunks de cache
- `LMCACHE_LOCAL_CPU=True`: armazena cache na memória CPU
- `LMCACHE_MAX_LOCAL_CPU_SIZE=150.0`: até 150GB de cache em CPU
- `kv_connector=LMCacheConnectorV1`: integração com vLLM

In [ ]:
!pkill -f "vllm serve"

In [ ]:
%%bash

export LMCACHE_USE_EXPERIMENTAL=True
export LMCACHE_CHUNK_SIZE=256
export LMCACHE_LOCAL_CPU=True
export LMCACHE_MAX_LOCAL_CPU_SIZE=150.0
export LMCACHE_USE_LAYERWISE=False

# Inicia o servidor vLLM com LMCache habilitado
nohup vllm serve \
  Qwen/Qwen3-14B \
  --disable-log-requests \
  --kv-transfer-config '{"kv_connector":"LMCacheConnectorV1","kv_role":"kv_both"}' \
  --max-model-len 40960 \
  --gpu-memory-utilization 0.9 \
  > vllm.log 2>&1 &

In [ ]:
check_vllm_status()

### Configuração do cliente OpenAI

O vLLM expõe uma API compatível com OpenAI, permitindo usar o client padrão.

In [ ]:
from openai import OpenAI
from vllm import SamplingParams

# Configura o cliente para usar o servidor vLLM local
openai_api_key = "EMPTY"
openai_api_base = "http://localhost:8000/v1"
client = OpenAI(
    api_key=openai_api_key,
    base_url=openai_api_base,
)
model_id = "Qwen/Qwen3-14B"

sampling_params = SamplingParams(temperature=0, top_p=0.95, max_tokens=10)

def print_output_online(client, prompt, sampling_params, req_str):
    """Envia prompt ao servidor e mede o tempo de resposta."""
    start = time.time()
    completion = client.completions.create(model=model_id, prompt=prompt)
    print("Resultado:", completion)
    print("-" * 50)
    time_elapsed = time.time() - start
    print(f"Geração levou {time_elapsed:.2f} segundos, requisição {req_str}.")
    print("-" * 50)
    return time_elapsed

### Teste: Cold vs Warm Cache

Enviamos 30 prompts com prefixos longos e repetidos. Na primeira rodada (cold), o cache está vazio. Na segunda rodada (warm), o cache já contém os KV dos prefixos, acelerando drasticamente o prefill.

In [ ]:
# Primeira rodada: cache frio (cold)
time_cold_lmcache = []
for i in range(30)[::-1]:
    print('Primeira vez', i)
    shared_prompt = f"Hello this is now a brand new prompt, how are you? {i}"
    auto_prompt = shared_prompt * 2000 + "Hello, my name is"
    time_cold_lmcache.append(print_output_online(client, auto_prompt, sampling_params, "auto"))

# Segunda rodada: cache quente (warm)
time_warm_lmcache = []
for i in range(30):
    print('Segunda vez', i)
    shared_prompt = f"Hello this is now a brand new prompt, how are you? {i}"
    auto_prompt = shared_prompt * 2000 + "Hello, my name is"
    time_warm_lmcache.append(print_output_online(client, auto_prompt, sampling_params, "auto"))

### Baseline: vLLM sem LMCache

Agora reiniciamos o servidor sem LMCache para comparar os tempos.

In [ ]:
!pkill -f "vllm serve"
time.sleep(15)

In [ ]:
%%bash
nohup vllm serve \
  Qwen/Qwen3-14B \
  --disable-log-requests \
  --max-model-len 40960 \
  --gpu-memory-utilization 0.90 \
  > vllm.log 2>&1 &

In [ ]:
check_vllm_status()

In [ ]:
# Sem LMCache: cold
time_cold_vllm = []
for i in range(30)[::-1]:
    print('Primeira vez', i)
    shared_prompt = f"Hello this is now a brand new prompt, how are you? {i}"
    auto_prompt = shared_prompt * 2000 + "Hello, my name is"
    time_cold_vllm.append(print_output_online(client, auto_prompt, sampling_params, "auto"))

# Sem LMCache: warm
time_warm_vllm = []
for i in range(30):
    print('Segunda vez', i)
    shared_prompt = f"Hello this is now a brand new prompt, how are you? {i}"
    auto_prompt = shared_prompt * 2000 + "Hello, my name is"
    time_warm_vllm.append(print_output_online(client, auto_prompt, sampling_params, "auto"))

### Visualização: LMCache vs vLLM padrão

Comparação do tempo de execução por iteração, mostrando o impacto do cache de KV.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

iterations = np.arange(1, len(time_cold_lmcache) + 1)

# Comparação Cold
plt.figure(figsize=(10, 6))
plt.plot(iterations, time_cold_vllm, marker='o', label='vLLM (cold)', linewidth=2)
plt.plot(iterations, time_cold_lmcache, marker='s', label='LMCache (cold)', linewidth=2)
plt.ylim(bottom=0)
plt.xlabel("Iteração")
plt.ylabel("Tempo de Execução (s)")
plt.title("Comparação de Tempo: Cold Cache")
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()

# Comparação Warm
plt.figure(figsize=(10, 6))
plt.plot(iterations, time_warm_vllm, marker='o', label='vLLM (warm)', linewidth=2)
plt.plot(iterations, time_warm_lmcache, marker='s', label='LMCache (warm)', linewidth=2)
plt.ylim(bottom=0)
plt.xlabel("Iteração")
plt.ylabel("Tempo de Execução (s)")
plt.title("Comparação de Tempo: Warm Cache")
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()

---
## 6.2 — Speculative Decoding: Gerando Tokens Mais Rápido

O **Speculative Decoding** usa um modelo menor (draft) ou heurísticas (n-gram) para gerar vários tokens candidatos de uma vez, que são verificados pelo modelo principal em uma única passada. Se os candidatos estiverem corretos, vários tokens são aceitos simultaneamente.

Variantes testadas:
1. **Vanilla vLLM** — Baseline sem especulação
2. **N-gram** — Usa padrões de n-gram do prompt para especular
3. **N-gram melhorado** — N-gram com janela maior de lookup
4. **Eagle3** — Modelo draft treinado especificamente para especulação

In [ ]:
!uv pip install vllm==0.11.0 torch

In [ ]:
import vllm
from importlib.metadata import version
print(f"vllm: {version('vllm')}")
print(f"torch: {version('torch')}")

### Função de benchmark com Spec-Bench

Usamos o dataset [Spec-Bench](https://github.com/hemingkx/Spec-Bench) para avaliar speculative decoding de forma padronizada.

In [ ]:
from openai import OpenAI

openai_api_key = "EMPTY"
openai_api_base = "http://localhost:8000/v1"
client = OpenAI(api_key=openai_api_key, base_url=openai_api_base)
model_id = "Qwen/Qwen3-32B"

def run_vllm_bench_spec(
    *,
    base_url="http://localhost:8000",
    model="Qwen/Qwen3-32B",
    num_prompts=16,
    spec_bench_output_len=512,
    spec_bench_category=None,
    dataset_path="Spec-Bench/data/spec_bench/question.jsonl",
    max_concurrency=1,
    request_rate=None,
    burstiness=None,
    seed=None,
    result_filename="spec_bench_decode.json",
    save_result=True,
    append_result=True,
):
    """Executa benchmark de speculative decoding via Spec-Bench."""
    cmd_parts = [
        "vllm", "bench", "serve",
        "--backend", "vllm",
        "--base-url", base_url,
        "--endpoint", "/v1/completions",
        "--dataset-name", "spec_bench",
        "--num-prompts", str(num_prompts),
        "--model", model,
        "--max-concurrency", str(max_concurrency),
        "--spec-bench-output-len", str(spec_bench_output_len),
    ]

    if dataset_path is None:
        raise ValueError("Spec Bench precisa de --dataset-path")
    cmd_parts += ["--dataset-path", dataset_path]

    if spec_bench_category:
        cmd_parts += ["--spec-bench-category", spec_bench_category]
    if request_rate is not None:
        cmd_parts += ["--request-rate", str(request_rate)]
    if burstiness is not None:
        cmd_parts += ["--burstiness", str(burstiness)]
    if seed is not None:
        cmd_parts += ["--seed", str(seed)]
    if save_result:
        cmd_parts.append("--save-result")
    if append_result:
        cmd_parts.append("--append-result")
    if result_filename:
        cmd_parts += ["--result-filename", result_filename]

    print("Executando:", " ".join(cmd_parts))

    import subprocess
    process = subprocess.Popen(
        cmd_parts,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    try:
        for line in process.stdout:
            print(line, end="")
    except KeyboardInterrupt:
        print("Interrompido pelo usuário...")
        process.terminate()
    process.wait()
    print(f"\nBenchmark finalizado com código {process.returncode}")
    return process.returncode

In [ ]:
!git clone https://github.com/hemingkx/Spec-Bench.git

### Variante 1: Vanilla vLLM (baseline sem especulação)

In [ ]:
%%bash
nohup vllm serve Qwen/Qwen3-32B \
  --disable-log-requests \
  --max-model-len 2048 \
  --gpu-memory-utilization 0.95 \
  > vllm.log 2>&1 &

In [ ]:
check_vllm_status()
run_vllm_bench_spec(max_concurrency=1)
run_vllm_bench_spec(max_concurrency=16)

### Variante 2: N-gram Speculative Decoding

Usa padrões de n-gram encontrados no prompt para especular os próximos tokens. Funciona bem quando o prompt contém texto que o modelo tende a repetir ou parafrasear.

In [ ]:
!pkill -f "vllm serve"
time.sleep(15)

In [ ]:
%%bash
nohup vllm serve Qwen/Qwen3-32B \
  --speculative-config '{
    "method": "ngram",
    "num_speculative_tokens": 6,
    "prompt_lookup_min": 4,
    "prompt_lookup_max": 6
}' \
  --disable-log-requests \
  --max-model-len 2048 \
  --gpu-memory-utilization 0.95 \
  > vllm.log 2>&1 &

In [ ]:
check_vllm_status()
run_vllm_bench_spec(max_concurrency=1)
run_vllm_bench_spec(max_concurrency=16)

### Variante 3: N-gram Melhorado

Ampliamos a janela de lookup (2 a 128 tokens) para capturar padrões maiores.

In [ ]:
!pkill -f "vllm serve"
time.sleep(15)

In [ ]:
%%bash
nohup vllm serve Qwen/Qwen3-32B \
  --speculative-config '{
    "method": "ngram",
    "num_speculative_tokens": 4,
    "prompt_lookup_min": 2,
    "prompt_lookup_max": 128
}' \
  --disable-log-requests \
  --max-model-len 2048 \
  --gpu-memory-utilization 0.95 \
  > vllm.log 2>&1 &

In [ ]:
check_vllm_status()
run_vllm_bench_spec(max_concurrency=1)
run_vllm_bench_spec(max_concurrency=16)

### Variante 4: Eagle3

O Eagle3 usa um modelo "speculator" treinado especificamente para prever múltiplos tokens futuros. É a abordagem mais sofisticada e geralmente oferece a melhor taxa de aceitação.

In [ ]:
!pkill -f "vllm serve"
time.sleep(15)

In [ ]:
%%bash
nohup vllm serve Qwen/Qwen3-32B \
  --speculative-config '{
    "method": "eagle3",
    "model": "RedHatAI/Qwen3-32B-speculator.eagle3",
    "num_speculative_tokens": 3
  }' \
  --disable-log-requests \
  --max-model-len 2048 \
  --gpu-memory-utilization 0.95 \
  > vllm.log 2>&1 &

In [ ]:
check_vllm_status()
run_vllm_bench_spec(max_concurrency=1)
run_vllm_bench_spec(max_concurrency=16)

### Visualização Comparativa das 4 Variantes

Comparamos throughput, duração e latência (TTFT vs TPOT) entre todas as variantes.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

# Carrega dados do benchmark
with open("spec_bench_decode.json", "r") as f:
    data = [json.loads(line) for line in f if line.strip()]

variant_labels = ['vanilla vllm', 'n-gram', 'n-gram melhorado', 'eagle3']

# Detecta concorrências
concs = sorted({d["max_concurrency"] for d in data})
c1, c2 = concs

# Organiza registros por variante e concorrência
records = {}
for i, d in enumerate(data):
    v_idx = i // 2
    records[(v_idx, d["max_concurrency"])] = d

def metric_by_concurrency(key, scale=1.0):
    vals_c1 = [records[(i, c1)][key] / scale for i in range(len(variant_labels))]
    vals_c2 = [records[(i, c2)][key] / scale for i in range(len(variant_labels))]
    return np.array([vals_c1, vals_c2])

total_tput = metric_by_concurrency("total_token_throughput")
output_tput = metric_by_concurrency("output_throughput")
durations = metric_by_concurrency("duration")
mean_ttft = metric_by_concurrency("mean_ttft_ms")
mean_tpot = metric_by_concurrency("mean_tpot_ms")

def grouped_bars_by_variant(ax, values, title, ylabel, variants, concurrencies):
    num_conc, num_vars = values.shape
    x = np.arange(num_conc)
    group_width = 0.85
    bar_width = group_width / num_vars
    offsets = (np.arange(num_vars) - (num_vars - 1) / 2.0) * bar_width

    for v_idx in range(num_vars):
        ax.bar(x + offsets[v_idx], values[:, v_idx], width=bar_width, label=variants[v_idx])
        for xi, val in zip(x + offsets[v_idx], values[:, v_idx]):
            ax.text(xi, val * 1.02, f"{val:.1f}", ha='center', va='bottom', fontsize=8)

    ax.set_xticks(x)
    ax.set_xticklabels([str(c) for c in concurrencies])
    ax.set_xlabel("Concorrência")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(axis='y', linestyle='--', alpha=0.6)
    ax.legend(ncols=min(len(variants), 4), fontsize=9)

# Total Token Throughput
fig, ax = plt.subplots(figsize=(9, 5))
grouped_bars_by_variant(ax, total_tput, "Throughput Total de Tokens", "Tokens/s", variant_labels, [c1, c2])
plt.tight_layout(); plt.show()

# Output Throughput
fig, ax = plt.subplots(figsize=(9, 5))
grouped_bars_by_variant(ax, output_tput, "Throughput de Saída", "Tokens/s", variant_labels, [c1, c2])
plt.tight_layout(); plt.show()

# Duração
fig, ax = plt.subplots(figsize=(9, 5))
grouped_bars_by_variant(ax, durations, "Duração da Execução", "Segundos", variant_labels, [c1, c2])
plt.tight_layout(); plt.show()

# Latência: TTFT vs TPOT (concorrência 16)
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(variant_labels))
bar_width = 0.35
ttft_vals = mean_ttft[1, :]
tpot_vals = mean_tpot[1, :]
plt.bar(x - bar_width/2, ttft_vals, width=bar_width, label='TTFT Médio (ms)', color='steelblue')
plt.bar(x + bar_width/2, tpot_vals, width=bar_width, label='ITL Médio (ms)', color='darkorange')
for pos, val in zip(x - bar_width/2, ttft_vals):
    plt.text(pos, val * 1.02, f"{val:.1f}", ha='center', va='bottom', fontsize=8)
for pos, val in zip(x + bar_width/2, tpot_vals):
    plt.text(pos, val * 1.02, f"{val:.1f}", ha='center', va='bottom', fontsize=8)
plt.xticks(x, variant_labels)
plt.ylabel("Latência (ms)")
plt.title("TTFT Médio vs ITL Médio (Concorrência = 16)")
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()